In [ ]:
# 基于之前通用模型的关键特征进行处理，首先确定关键特征有哪些
# Plan A

In [ ]:
# Plan A top6 Feature SR

In [ ]:
import pandas as pd
import deap
import operator
import math
import random
from deap import algorithms, base, creator, gp, tools
import numpy as np
from sklearn.metrics import mean_squared_error  # You can adjust this if you're working with regression metrics

origin_feature_file = '../../shap_new/index/qf_final_feature_shap_values.xlsx'
dimension = 2
YS_Feature1 = pd.read_excel(origin_feature_file)['Feature Name'][:dimension]
Target = '屈服强度'  # Yield Strength
print(f'overall {len(YS_Feature1)} features','\n',YS_Feature1)
print(f'overall {len(YS_Feature1)} features')

train_data_file = '../../Tensile_Strength/train_set_new.xlsx'

# Reading training data
X = pd.read_excel(train_data_file, header=0)[YS_Feature1]  
Y = pd.read_excel(train_data_file, header=0)[Target] 

# Create the primitive set (set of operations)
pset = gp.PrimitiveSet("MAIN", dimension)  # dimension=2 means two inputs
pset.addPrimitive(operator.add, 2)
pset.addPrimitive(operator.sub, 2)
pset.addPrimitive(operator.mul, 2)
pset.addPrimitive(operator.truediv, 2)  # Division
pset.addPrimitive(operator.neg, 1)  # Negation
pset.addPrimitive(math.sin, 1)  # Sine function
pset.addPrimitive(math.cos, 1)  # Cosine function
pset.addPrimitive(math.exp, 1)  # Exponential function
pset.addPrimitive(math.log, 1)  # Logarithmic function
pset.addEphemeralConstant("rand", lambda: random.uniform(-1, 1))  # Random constant
pset.renameArguments(ARG0="x0", ARG1='x1')  # Rename variables for better readability

# Create fitness function and individual class
creator.create("FitnessMin", base.Fitness, weights=(-1.0,))  # Minimize error (for regression)
creator.create("Individual", gp.PrimitiveTree, fitness=creator.FitnessMin)

# Toolbox setup
toolbox = base.Toolbox()
toolbox.register("expr", gp.genFull, pset=pset, min_=1, max_=3)  # Generate full expressions with depth 1-3
toolbox.register("individual", tools.initIterate, creator.Individual, toolbox.expr)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

def evaluate(individual):
    """Evaluate the fitness of an individual."""
    # Compile the individual into a function using gp.compile
    func = gp.compile(expr=individual, pset=pset)
    
    # Apply the function to the input data (X)
    predictions = np.array([func(*x) for x in X.values])
    
    # Use Mean Squared Error (MSE) for regression fitness
    mse = mean_squared_error(Y, predictions)
    return mse,



# Register the evaluation function in the toolbox
toolbox.register("evaluate", evaluate)

# Selection, crossover, and mutation functions
toolbox.register("select", tools.selTournament, tournsize=3)  # Tournament selection
toolbox.register("mate", gp.cxOnePoint)  # One-point crossover
toolbox.register("mutate", gp.mutUniform, expr=toolbox.expr, min_=1, max_=3)  # Uniform mutation
toolbox.register("map", map)  # Use map for parallel processing

# Set up the population size and number of generations
population_size = 50
generations = 40
cx_prob = 0.7  # Crossover probability
mut_prob = 0.2  # Mutation probability

# Create initial population
population = toolbox.population(n=population_size)

# Define statistics to track the best individual in each generation
stats = tools.Statistics(lambda ind: ind.fitness.values)
stats.register("avg", np.mean)
stats.register("min", np.min)
stats.register("max", np.max)

# Run the evolutionary algorithm
result = algorithms.eaSimple(population, toolbox, cxpb=cx_prob, mutpb=mut_prob, 
                             ngen=generations, stats=stats, halloffame=None, verbose=True)

# Extract the best individual and its fitness
best_individual = tools.selBest(population, 1)[0]
print("Best individual is:", best_individual)
print("Fitness value is:", best_individual.fitness.values[0])

# Optional: Save the best individual’s expression
best_expr = toolbox.compile(expr=best_individual)
print("Best expression:", best_expr)



In [50]:
YS_Feature1 = pd.read_excel(origin_feature_file)['Feature Name'][:10]
print(list(YS_Feature1))
TS_Feature1 = pd.read_excel('../../shap_new/index/kl_final_feature_shap_values.xlsx')['Feature Name'][:10]
print(list(TS_Feature1))

['Calculated Yield Strength', 'Grain Size', 'Habit Plane', 'Calculated Grain Boundary', 'Interant electrons', 'frac f valence electrons', 'Zr fraction', 'MagpieData avg_dev MeltingT', 'mean AtomicRadius', 'Yang omega']
['Grain Size', 'Habit Plane', 'Calculated Yield Strength', 'MagpieData mean GSvolume_pa', 'Interant electrons', 'Interant d electrons', 'Calculated Grain Boundary', 'MagpieData range GSvolume_pa', 'Zr fraction', 'Mixing enthalpy']


In [ ]:
import pandas as pd
import deap
import operator
import math
import random
from deap import algorithms, base, creator, gp, tools
import numpy as np
from sklearn.metrics import mean_squared_error  # You can adjust this if you're working with regression metrics

origin_feature_file = '../../shap_new/index/qf_final_feature_shap_values.xlsx'

print(f'overall {len(YS_Feature1)} features','\n',YS_Feature1)
print(f'overall {len(YS_Feature1)} features')

train_data_file_qf = '../../Tensile_Strength/train_set_new.xlsx'
train_data_file_kl = '../../Tensile_Strength/train_set_new.xlsx'
task_list =[f'taks{i}' for i in range(1,5)]
def get_dataset_qf(data_file,task):
    Target='屈服强度'
    if task =='task1':
        # 选择前4个特征,不包含fraction和habit plane
        names = ['Calculated Yield Strength', 'Grain Size', 'Calculated Grain Boundary', 'Interant electrons']
        Feature = pd.read_excel(data_file)['Feature Name'][names]        
        print('task:',task,Feature)
    if task =='task2':
         # 选择前6个特征,不包含fraction和habit plane
        names = ['Calculated Yield Strength', 'Grain Size', 'Calculated Grain Boundary', 'Interant electrons', 'frac f valence electrons', 'MagpieData avg_dev MeltingT']
        Feature = pd.read_excel(data_file)['Feature Name'][names]
        print('task:',task,Feature)
    if task =='task3':
        # 选择前4个特征,不包含计算强度
        names = ['Grain Size','Interant electrons', 'frac f valence electrons', 'MagpieData avg_dev MeltingT']        
        Feature = pd.read_excel(data_file)['Feature Name'][names]
        print('task:',task,Feature)        
    if task =='task4':
        # 选择前6个特征,不包含计算强度
        names = ['Grain Size','Interant electrons', 'frac f valence electrons', 'MagpieData avg_dev MeltingT', 'mean AtomicRadius', 'Yang omega']        
        Feature = pd.read_excel(data_file)['Feature Name'][names]
        print('task:',task,Feature)    
    X = pd.read_excel(data_file, header=0)[Feature]  # 确保列标题被正确读取
    Y = pd.read_excel(data_file, header=0)[Target]
    return X, Y

def get_dataset_kl(data_file,task):
    Target='抗拉强度 (UTS)'
    if task =='task1':
        # 选择前4个特征,不包含fraction和habit plane
        names = ['Grain Size', 'Calculated Yield Strength', 'MagpieData mean GSvolume_pa', 'Interant electrons']       
        Feature = pd.read_excel(data_file)['Feature Name'][names]
        Feature = pd.read_excel(data_file)['Feature Name'][names]        
        print('task:',task,Feature)
    if task =='task2':
         # 选择前6个特征,不包含fraction和habit plane
        names = ['Grain Size', 'Calculated Yield Strength', 'MagpieData mean GSvolume_pa', 'Interant electrons', 'Interant d electrons', 'Calculated Grain Boundary']       
        Feature = pd.read_excel(data_file)['Feature Name'][names]
        print('task:',task,Feature)
    if task =='task3':
        # 选择前4个特征,不包含计算强度
        names = ['Grain Size',  'MagpieData mean GSvolume_pa', 'Interant electrons', 'Interant d electrons']       
        Feature = pd.read_excel(data_file)['Feature Name'][names]
        Feature = pd.read_excel(data_file)['Feature Name'][names]
        print('task:',task,Feature)        
    if task =='task4':
        # 选择前6个特征,不包含计算强度
        names = ['Grain Size',  'MagpieData mean GSvolume_pa', 'Interant electrons', 'Interant d electrons', 'MagpieData range GSvolume_pa', 'Mixing enthalpy']       
        Feature = pd.read_excel(data_file)['Feature Name'][names]
        print('task:',task,Feature)    
    X = pd.read_excel(data_file, header=0)[Feature]  # 确保列标题被正确读取
    Y = pd.read_excel(data_file, header=0)[Target]
    return X, Y



In [ ]:
!pip install pysr

In [ ]:
from pysr import PySRRegressor
from sympy import cos  # Import SymPy for custom operator definitions

model = PySRRegressor(
    maxsize=50,  # Maximum complexity
    niterations=100,  # Number of iterations
    binary_operators=["*", "+", "-", "/"],
    unary_operators=["square", "cube", "exp", "cos2(x)=cos(x)^2"],
    constraints={
        "/": (-1, 9),
        "square": 9,
        "cube": 9,
        "exp": 9,
    },
    nested_constraints={
        "square": {"square": 1, "cube": 1, "exp": 0},
        "cube": {"square": 1, "cube": 1, "exp": 0},
        "exp": {"square": 1, "cube": 1, "exp": 0},
    },
    extra_sympy_mappings={
        "inv": lambda x: 1 / x,  # Inverse operator
        "cos2": lambda x: cos(x)**2,  # Custom operator for `cos2`
    },
    elementwise_loss="loss(prediction, target) = (prediction - target)^2",
    early_stop_condition=(
        "stop_if(loss, complexity) = loss < 1e-6 && complexity < 10"
    ),
    timeout_in_seconds=60 * 60 * 2,  # Timeout in seconds
)

X = pd.read_excel(train_data_file, header=0)[YS_Feature1]  # 确保列标题被正确读取
Y = pd.read_excel(train_data_file, header=0)[Target]

print("X data preview:", X.head())  # 查看X的前几行
print("Y data preview:", Y.head())  # 查看Y的前几行
model.fit(X, Y)
print(model)


[juliapkg] Found dependencies: d:\anaconda\envs\mg_alloy\lib\site-packages\pysr\juliapkg.json
[juliapkg] Found dependencies: d:\anaconda\envs\mg_alloy\lib\site-packages\juliapkg\juliapkg.json
[juliapkg] Found dependencies: d:\anaconda\envs\mg_alloy\lib\site-packages\juliacall\juliapkg.json
[juliapkg] Locating Julia =1.10.0, ^1.10.3
[juliapkg] Querying Julia versions from https://julialang-s3.julialang.org/bin/versions.json
[juliapkg] WARNING: About to install Julia 1.11.2 to D:\Anaconda\envs\Mg_Alloy\julia_env\pyjuliapkg\install.
[juliapkg]   If you use juliapkg in more than one environment, you are likely to
[juliapkg]   have Julia installed in multiple locations. It is recommended to
[juliapkg]   install JuliaUp (https://github.com/JuliaLang/juliaup) or Julia
[juliapkg]   (https://julialang.org/downloads) yourself.
[juliapkg] Downloading Julia from https://julialang-s3.julialang.org/bin/winnt/x64/1.11/julia-1.11.2-win64.zip
             downloaded 47.2 MB of 260.5 MB
             dow

d:\Anaconda\envs\Mg_Alloy\lib\site-packages\pysr\sr.py:2727: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
d:\Anaconda\envs\Mg_Alloy\lib\site-packages\pysr\sr.py:1529: UserWarning: Spaces in DataFrame column names are not supported. Spaces have been replaced with underscores. 
Please rename the columns to valid names.
  warnings.warn(
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           6.762e+03  1.594e+01  y = Calculated_Yield_Strength
3           5.421e+03  1.105e-01  y = Calculated_Yield_Strength + 36.618
5           4.245e+03  1.223e-01  y = (Calculated_Yield_Strength * 0.60812) + 96.208
7           4.205e+03  4.758e-03  y = exp((inv(Calculated_Yield_Strength) * -72.637) + 5.815...
                                      5)
9           4.098e+03  1.279e-02  y = (Calculated_Yield_Strength * ((Calculated_Yield_Streng...
                                      th * -0.00094742) + 1.0273)) + 61.623
10          4.095e+03  8.964e-04  y = (Calculated_Yield_Strength * (sin(Calculated_Yield_Str...
                                      ength * -0.00097107) + 1.0275)) + 61.147
11          4.057e+03  9.299e-03  y = (Calculated_Yield_Strength + (Calculated_Yield_Strengt...
                                      h * ((Calculated_

In [ ]:
def get_dataset(data_file,task):
    if task =='task1':
        YS_Feature = 
        Target = '屈服强度'
    X = pd.read_excel(data_file, header=0)[YS_Feature]  # 确保列标题被正确读取
    Y = pd.read_excel(data_file, header=0)[Target]
    return X, Y

In [30]:
print("X shape:", X.shape)
print("Y shape:", Y.shape)
print("X data preview:", X.head())  # 如果 X 是 DataFrame
print("Y data preview:", Y.head())  # 如果 Y 是 DataFrame


X shape: (739, 2)
Y shape: (739,)
X data preview:    Calculated Yield Strength  Grain Size
0                 226.991971         4.0
1                 121.106646        36.0
2                 179.402426        50.0
3                 353.658570         1.5
4                 113.089422        15.0
Y data preview: 0    364.0
1    125.0
2    200.0
3    290.0
4    148.0
Name: 屈服强度, dtype: float64


In [23]:
# print(X.dtypes,Y.dtypes)

X = pd.read_excel(train_data_file)[YS_Feature1]  
Y = pd.read_excel(train_data_file)[Target] 
Y_numeric = pd.to_numeric(Y, errors='coerce')
problematic_rows = Y_numeric.isna()
print("Problematic rows:")
print(Y[problematic_rows])  # 显示含有非数值的行




Problematic rows:
Series([], Name: 屈服强度, dtype: float64)


In [ ]:


# 生成样本数据
import numpy as np
X = np.linspace(-1, 1, 100)  # 输入数据
Y = np.array([target_function(x) for x in X])  # 目标输出

